<a href="https://bio-pro.mintlify.app/tools/guides/parallel-execution"><img align="right" src="https://img.shields.io/badge/View_in_Proto_Docs_%E2%86%92-046e7a?style=for-the-badge&logo=readthedocs&logoColor=white" alt="View in Proto Docs: Parallel Execution"></a>

# 04 — Parallel Execution with ToolPool

> **Local inference only.** This notebook covers how `proto_tools` manages models on your own hardware. If you're running via **cloud inference**, this is all handled for you — skip ahead to the next feature.

> Prerequisites: [02 — Tool Persistence](02_tool_persistence.ipynb) and [03 — Device Management](03_device_management.ipynb).

`ToolPool` intercepts `@tool`-decorated calls transparently, partitions iterable inputs across your GPUs, runs them in parallel, and reassembles the results in the original order. **No changes to tool code** — just wrap your calls in a `ToolPool()` context.

**Features**

- **Transparent interception** — any tool with an iterable input field is auto-partitioned and parallelized
- **Cost-aware scheduling** — items are distributed via LPT bin-packing using each tool's `item_cost()` estimate
- **Built-in persistence** — workers stay alive across calls within the pool (no reload cost)
- **Device auto-detection** — discovers all visible GPUs, or accepts an explicit device list
- **Automatic dedup** — duplicate items in the input are computed once and expanded back to their original positions

You need **2 or more GPUs** to see the speedup in this notebook.


In [ ]:
from time import time

from proto_tools.tools.structure_prediction.esmfold import (
    ESMFoldInput,
    run_esmfold,
)
from proto_tools.utils import display_gpu_memory_usage
from proto_tools.utils.device_manager import DeviceManager
from proto_tools.utils.tool_instance import ToolInstance
from proto_tools.utils.tool_pool import ToolPool

## A varied batch of sequences

We fold 48 sequences of varied lengths so the LPT scheduler has something to work with (short sequences pair with long ones to balance each GPU's workload).


In [ ]:
# Base fragments — vary lengths so LPT scheduling has something to work with
_FRAGMENTS = [
    # ~15 residues (short)
    "MKTLLILAVVAAALA",
    "GAVLTVLLGGLLLA",
    "MGQQPGKVLGDQRR",
    "ASTVKFLGPVLDAA",
    # ~75 residues (medium)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNP",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    # ~150 residues (long)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNP",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAKGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    # ~300 residues (extra long)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAKMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
]

# 3x replication with single-residue suffixes to make each copy unique
# (avoids @tool dedup so the benchmark measures pure parallelism)
SEQUENCES = (
    [s + "A" for s in _FRAGMENTS]   # batch 1
    + [s + "G" for s in _FRAGMENTS] # batch 2
    + [s + "V" for s in _FRAGMENTS] # batch 3
)

print(f"{len(SEQUENCES)} sequences (all unique)")
print(f"Length distribution: {sorted(set(len(s) for s in SEQUENCES))}")
print(f"Total residues: {sum(len(s) for s in SEQUENCES):,}")

---
## 1. Basic usage — auto-detect GPUs

The simplest way to use `ToolPool`: just wrap your tool calls. It auto-detects every visible GPU and partitions the input across them.

Both runs below use **warm** workers (model already loaded) so we measure pure inference throughput, not cold-start overhead.


In [ ]:
complexes = [[seq] for seq in SEQUENCES]

# --- Sequential: single GPU, warm worker ---
print("=== Sequential (1 GPU, warm) ===")
with ToolInstance.persist():
    t0 = time()
    result_seq = run_esmfold(ESMFoldInput(complexes=complexes))
    sequential_time = time() - t0
    print(f"Time: {sequential_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

print("\n(cleared)")
display_gpu_memory_usage()

# --- Parallel: split across all GPUs, warm workers ---
print("\n=== Parallel (ToolPool, auto-detect GPUs) ===")
with ToolPool():
    t0 = time()
    result_par = run_esmfold(ESMFoldInput(complexes=complexes))
    parallel_time = time() - t0
    print(f"Time: {parallel_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

print(f"\nSpeedup: {sequential_time / parallel_time:.2f}x")
print(f"Structures returned in order: {len(result_par.structures)} (same as input)")

print("\n(cleaned up)")
display_gpu_memory_usage()

---
## 2. Persistence across calls

Workers stay alive across calls within the same `ToolPool` context. The first call pays the model-loading cost on every GPU; subsequent calls run at full speed.


In [ ]:
complexes = [[seq] for seq in SEQUENCES]

print("=== Persistence demo: cold vs warm calls ===")
with ToolPool(devices=["cuda:0", "cuda:1"]):
    # Cold call — pays model loading on both GPUs
    t0 = time()
    result1 = run_esmfold(ESMFoldInput(complexes=complexes))
    cold_time = time() - t0
    print(f"Cold call: {cold_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

    # Warm call — workers already loaded, pure inference
    t0 = time()
    result2 = run_esmfold(ESMFoldInput(complexes=complexes))
    warm_time = time() - t0
    print(f"\nWarm call: {warm_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

    # Another warm call to confirm consistency
    t0 = time()
    result3 = run_esmfold(ESMFoldInput(complexes=complexes))
    warm_time2 = time() - t0
    print(f"\nWarm call 2: {warm_time2:.1f}s for {len(complexes)} structures")

print(f"\nCold → Warm speedup: {cold_time / warm_time:.2f}x")
print(f"Model loading overhead: ~{cold_time - warm_time:.0f}s")

print("\n(cleaned up)")
ToolInstance.clear_all()
DeviceManager.reset_instance()
display_gpu_memory_usage()

## Takeaways

- **`with ToolPool():`** is the easy button — zero code changes inside the block.
- **`with ToolPool(devices=[...])`** lets you restrict or explicitly pick GPUs.
- **First call in the block pays for model loading;** every call after is warm.
- **Dedup is automatic** — identical items in `complexes` are computed once and expanded back to all their positions in the output.

Curious about the internals (LPT scheduling, cost estimation)? See `utils/tool_pool.py`.
